# Medical Sector Prediction Model
## 申万医药生物指数 — 周频方向 + 幅度预测

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd; import numpy as np
import matplotlib.pyplot as plt; import seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# 1. Load data
from src.data_fetcher.akshare_source import AKShareSource
from src.data_fetcher.fred_source import FREDSource

data = {}
data.update(AKShareSource().fetch_all('2020-01-01'))
data['gold_futures'] = data.get('gold_concept_cn', pd.DataFrame())
data['gold_etf'] = data.get('gold_concept_cn', pd.DataFrame())
data.update(FREDSource().fetch_all('2020-01-01'))
print(f'Loaded {len(data)} data sources')

In [ ]:
# 2. Train model
from src.models.train import TrainingPipeline
from src.features.engineer import FeatureEngineer

fe = FeatureEngineer(lags=[1, 2, 4], rolling_windows=[4, 13], fill_method='ffill')
pipeline = TrainingPipeline('medical', data, feat_engineer=fe)
results = pipeline.run()

In [ ]:
# 3. Feature Importance (XGBoost built-in)
imp = pipeline.get_feature_importance()
fig, ax = plt.subplots(figsize=(12, 8))
top = imp.head(20).sort_values('importance')
ax.barh(range(len(top)), top['importance'].values, color='steelblue')
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top['feature'].values, fontsize=9)
ax.set_xlabel('Feature Importance')
ax.set_title('Medical Model — Top 20 Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# 4. SHAP Analysis
import shap

explainer = shap.TreeExplainer(pipeline.clf.model)
shap_values = explainer.shap_values(pipeline.X_test)

fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(shap_values, pipeline.X_test, show=False)
plt.tight_layout()
plt.show()

In [ ]:
# 5. Test Set Performance
from src.backtest.simple_backtest import SimpleBacktest, benchmark_comparison

y_pred_cls = pipeline.clf.predict(pipeline.X_test)
y_pred_reg = pipeline.reg.predict(pipeline.X_test)

bt = SimpleBacktest(
    predictions=pd.Series(y_pred_reg, index=pipeline.y_test.index),
    returns=pipeline.y_test['magnitude'],
    name='Medical XGBoost'
)

# Layered performance
layered = bt.layered_performance(n_groups=5)
print('Layered Performance:')
display(layered)

# Plot
bt.plot_layered()
plt.show()

# Benchmark comparison
bench = benchmark_comparison(
    model_hit=results['reg_test']['hit_ratio'],
    model_mae=results['reg_test']['mae'],
    target_returns=pipeline.y_test['magnitude']
)
print('\nBenchmark Comparison:')
for k, v in bench.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# 6. Market Regime Analysis
regime_df = bt.by_market_regime()
print('Performance by Market Regime:')
display(regime_df)

# 6. Signal Monotonicity
mono = bt.signal_monotonicity()
print(f'\nSignal Monotonicity: rho={mono["monotonic_rho"].iloc[0]:.4f}, p={mono["monotonic_pval"].iloc[0]:.4f}')

In [ ]:
# 7. 保存模型
pipeline.save_models()
print('Models saved to data/processed/')